# Imports and seed setting

In [1]:
%pip install vitaldb

In [2]:
import gc
import logging
import math
import os
import random
import time
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import signal as scipy_signal
from scipy.signal import butter, filtfilt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

In [4]:
# %pip install vitaldb

In [5]:
try:
    import vitaldb
    VITALDB_AVAILABLE = True
except ImportError:
    VITALDB_AVAILABLE = False
    warnings.warn("vitaldb not installed — data loading will be skipped.")


In [6]:
# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False  # sacrifice speed for reproducibility

# CONFIG dict — ALL hyperparameters, track names, file paths

In [7]:
CONFIG: Dict = {

    # ── Data ──────────────────────────────────────────────────────────────────
    "case_id_range": (1, 200),           # inclusive range for VitalDB case IDs
    "min_case_duration_sec": 1800,        # discard cases shorter than 30 min
    "min_sqi_coverage_frac": 0.70,        # ≥70% of case duration must have SQI≥70
    "sqi_hard_gate": 70,                  # exclude windows from loss if SQI < this
    "sqi_zero_weight": 50,                # zero-weight EEG embedding if SQI < this
    "max_cases": None,                    # set integer to cap loading (debug)

    # ── Tracks ────────────────────────────────────────────────────────────────
    "eeg_tracks": ["BIS/EEG1_WAV", "BIS/EEG2_WAV"],
    "eeg_interval": 1 / 128,             # 128 Hz
    "bis_tracks": [
        "BIS/BIS", "BIS/SQI", "BIS/EMG",
        "BIS/SR", "BIS/SEF", "BIS/TOTPOW"
    ],
    "bis_interval": 1,                   # 1 Hz
    "map_track": "Solar8000/ART_MBP",    # for IOH label construction

     # ── Window / Stride ───────────────────────────────────────────────────────
    "history_sec": 1800,                  # 30-min look-back for 1-Hz embedding output
    "eeg_window_sec": 30,                 # WaveNet processes 30-s chunks of raw EEG
    "eeg_samples_per_window": 3840,      # 30 s × 128 Hz
    "stride_sec": 60,                     # 1-min stride between windows
    "nan_threshold": 0.20,               # discard window if >20% NaN in any channel

     # ── Spectral features ─────────────────────────────────────────────────────
    "stft_window_sec": 4,
    "stft_overlap": 0.5,
    "bands": {                            # frequency bands (Hz) for power features
        "delta": (0.5, 4),
        "theta": (4, 8),
        "alpha": (8, 13),
        "beta":  (13, 30),
    },
    "sef_percentile": 95,
    "bsr_amplitude_threshold_uv": 5,     # μV threshold for burst suppression
    "bsr_window_sec": 63,                # BS ratio computation window
    "fs_eeg": 128,                        # raw EEG sampling rate

    # ── WaveNet CNN ───────────────────────────────────────────────────────────
    "wavenet_layers": 16,
    "wavenet_kernel_size": 3,
    "wavenet_residual_channels": 64,
    "wavenet_skip_channels": 64,
    "wavenet_dilations": [1,2,4,8,16,32,64,128, 1,2,4,8,16,32,64,128],  # two cycles
    "wavenet_output_length": 512,        # CNN output steps before pooling

    # ── Transformer ───────────────────────────────────────────────────────────
    "d_model": 128,
    "n_heads": 4,
    "feedforward_dim": 256,
    "transformer_layers": 4,
    "dropout": 0.1,

    # ── MSM Pretraining ───────────────────────────────────────────────────────
    "msm_mask_frac": 0.20,
    "msm_span_sec": 0.5,                 # 0.5-s random spans
    "msm_span_samples": 64,             # 0.5 s × 128 Hz
    "msm_epochs": 20,
    "msm_lr": 1e-4,
    "msm_batch_size": 16,

    # ── Supervised fine-tuning ────────────────────────────────────────────────
    "finetune_epochs": 50,
    "finetune_lr": 5e-5,
    "finetune_batch_size": 16,
    "weight_decay": 1e-4,
    "grad_clip": 1.0,

     # ── IOH label ─────────────────────────────────────────────────────────────
    "ioh_threshold_mmhg": 65.0,
    "ioh_min_duration_sec": 60,
    "ioh_horizons": [300, 600, 900],     # 5, 10, 15 min

    # ── N/H label ─────────────────────────────────────────────────────────────
    "nh_label_smoothing": 0.15,
    "nh_focal_gamma": 2.0,
    "hr_nociception_threshold": 90,      # bpm
    "bis_nociception_threshold": 60,
    "bis_hypnosis_threshold": 65,
    "hr_rise_pct_threshold": 0.15,       # 15% rise signals hypnosis excess

     # ── Training ──────────────────────────────────────────────────────────────
    "val_frac": 0.10,
    "test_frac": 0.10,
    "num_workers": 4,
    "early_stopping_patience": 8,
    "scheduler_patience": 4,
    "scheduler_factor": 0.5,

    # ── Paths ─────────────────────────────────────────────────────────────────
    "checkpoint_dir": "checkpoints",
    "log_file": "model_3.log",
    "msm_checkpoint": "checkpoints/model_3_msm.pt",
    "best_checkpoint": "checkpoints/model_3_best.pt",
    "final_checkpoint": "checkpoints/model_3_final.pt",
}

os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Logging setup

In [8]:
def setup_logging(log_file: str) -> logging.Logger:
    """Configure logger writing to both console and file.
    Parameters
    ---------
    log_file : str
        Path to the log file.
    Returns
    ------
    logging.Logger
        Configured logger instance.
    """
    logger = logging.getLogger("model_3")
    logger.setLevel(logging.DEBUG)
    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", "%Y-%m-%d %H:%M:%S")
    fh = logging.FileHandler(log_file, mode="w")
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(fmt)
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    ch.setFormatter(fmt)
    logger.addHandler(fh)
    logger.addHandler(ch)
    return logger

logger = setup_logging(CONFIG["log_file"])
logger.info(f"Model 3 — EEG WaveNet+Transformer  |  device={DEVICE}")

2026-06-11 17:41:28 [INFO] Model 3 — EEG WaveNet+Transformer  |  device=cuda
INFO:model_3:Model 3 — EEG WaveNet+Transformer  |  device=cuda


# Clinical metadata loader

In [9]:
def load_clinical_metadata() -> Optional[pd.DataFrame]:
    """Load case-level clinical metadata from VitalDB.
    Returns
    ------
    pd.DataFrame or None
        DataFrame indexed by caseid with columns: anestart, aneend, opstart, opend, etc.
    """
    if not VITALDB_AVAILABLE:
        logger.warning("vitaldb unavailable — returning None for clinical metadata.")
        return None
    try:
        df = vitaldb.load_clinical_data()
        logger.info(f"Clinical metadata loaded: {len(df)} cases.")
        return df
    except Exception as e:
        logger.error(f"load_clinical_data failed: {e}")
        return None

# Waveform data loader (EEG-only tracks)

In [10]:
def load_eeg_case(caseid: int, config: Dict) -> Optional[Dict[str, np.ndarray]]:
    """Load raw EEG waveforms and BIS scalar tracks for one case.
    Keeps everything in RAM — no disk writes.
    Parameters
    ---------
    caseid : int
    config : Dict
    Returns
    ------
    dict with keys 'eeg' (T_500hz, 2), 'bis' (T_1hz, 6), 'map' (T_1hz,) or None on failure.
    """
    result: Dict[str, Optional[np.ndarray]] = {}
    # EEG waveforms at 128 Hz
    try:
        eeg_arr = vitaldb.load_case(caseid, config["eeg_tracks"], interval=config["eeg_interval"])
        result["eeg"] = eeg_arr  # (T_128hz, 2)
    except Exception as e:
        logger.warning(f"caseid {caseid}: EEG load failed — {e}")
        result["eeg"] = None
    # BIS scalars at 1 Hz
    bis_and_map = config["bis_tracks"] + [config["map_track"]]
    try:
        scalar_arr = vitaldb.load_case(caseid, bis_and_map, interval=config["bis_interval"])
        if scalar_arr is not None:
            result["bis"] = scalar_arr[:, :len(config["bis_tracks"])]  # (T, 6)
            result["map"] = scalar_arr[:, len(config["bis_tracks"])]    # (T,)
        else:
            result["bis"] = None
            result["map"] = None
    except Exception as e:
        logger.warning(f"caseid {caseid}: BIS/MAP load failed — {e}")
        result["bis"] = None
        result["map"] = None
    # Require at least EEG or BIS to continue
    if result["eeg"] is None and result["bis"] is None:
        return None
    return result


In [11]:
def is_case_eligible(case_result: Dict, clinical_df: Optional[pd.DataFrame],
                     caseid: int, config: Dict) -> bool:
    """Check minimum duration and SQI coverage for a case.
    Parameters
    ---------
    case_result : dict  — from load_eeg_case()
    clinical_df : pd.DataFrame or None
    caseid : int
    config : Dict
    Returns
    ------
    bool
    """
    # Duration check via BIS 1-Hz array length
    bis = case_result.get("bis")
    if bis is None:
        return False
    duration_sec = bis.shape[0]
    if duration_sec < config["min_case_duration_sec"]:
        return False
    # SQI coverage check (column 1 of BIS array = BIS/SQI)
    sqi_col = bis[:, 1]
    valid_sqi = np.nansum(sqi_col >= config["sqi_hard_gate"])
    coverage = valid_sqi / max(duration_sec, 1)
    if coverage < config["min_sqi_coverage_frac"]:
        return False
    return True

# IOH LABEL CONSTRUCTION

In [12]:
def max_run_length(arr: np.ndarray) -> int:
    """Return the length of the longest consecutive run of 1s in a binary array.
    Parameters
    ---------
    arr : np.ndarray
        1D binary array.
    Returns
    ------
    int
    """
    if arr.sum() == 0:
        return 0
    max_run = cur_run = 0
    for val in arr:
        if val:
            cur_run += 1
            max_run = max(max_run, cur_run)
        else:
            cur_run = 0
    return max_run

In [13]:
def make_ioh_label(
    map_series: np.ndarray,
    horizon_sec: int = 300,
    threshold: float = 65.0,
    min_duration_sec: int = 60,
) -> np.ndarray:
    """Binary IOH label: 1 if MAP < threshold sustained ≥ min_duration_sec within horizon.
    Parameters
    ---------
    map_series : np.ndarray   1D float array at 1 Hz
    horizon_sec : int         look-ahead window in seconds
    threshold : float         MAP threshold in mmHg
    min_duration_sec : int    minimum sustained duration for a positive event
    Returns
    ------
    np.ndarray  1D binary float32, same length as map_series
    """
    n = len(map_series)
    labels = np.zeros(n, dtype=np.float32)
    for t in range(n - horizon_sec):
        window = map_series[t: t + horizon_sec]
        below = (window < threshold).astype(int)
        if max_run_length(below) >= min_duration_sec:
            labels[t] = 1.0
    return labels

In [14]:
def build_ioh_labels(
    map_series: np.ndarray, config: Dict
) -> Dict[str, np.ndarray]:
    """Build IOH labels at all three horizons (5, 10, 15 min).
    Parameters
    ---------
    map_series : np.ndarray
    config : Dict
    Returns
    ------
    dict keys: 'ioh_5', 'ioh_10', 'ioh_15'
    """
    out = {}
    for h in config["ioh_horizons"]:
        key = f"ioh_{h // 60}"
        out[key] = make_ioh_label(
            map_series,
            horizon_sec=h,
            threshold=config["ioh_threshold_mmhg"],
            min_duration_sec=config["ioh_min_duration_sec"],
        )
    return out

# N/H LABEL CONSTRUCTION (noisy supervision)

In [15]:
def build_nh_labels(bis_arr: np.ndarray, config: Dict) -> np.ndarray:
    """Construct N/H 3-class labels from BIS scalars.
    Classes:
        0 = adequate
        1 = nociception_excess  (BIS<60 AND implied HR>90 — proxied from BIS/EMG)
        2 = hypnosis_excess     (rising BIS>65 AND BIS trend rising)
    This is noisy supervision — clinical heuristics, not ground truth.
    For production, EVENT track opioid annotations should supplement this.
    Parameters
    ---------
    bis_arr : np.ndarray  shape (T, 6) columns: BIS, SQI, EMG, SR, SEF, TOTPOW
    config : Dict
    Returns
    ------
    np.ndarray  int64 shape (T,) with values in {0, 1, 2}
    """
    T = bis_arr.shape[0]
    labels = np.zeros(T, dtype=np.int64)  # default adequate
    bis = bis_arr[:, 0]
    sqi = bis_arr[:, 1]
    emg = bis_arr[:, 2]  # EMG as nociception proxy
    window = 300  # 5-min look-back for trend
    for t in range(window, T):
        if sqi[t] < config["sqi_hard_gate"]:
            continue  # unreliable window — leave as adequate (0)
        bis_val = bis[t]
        emg_val = emg[t] if not np.isnan(emg[t]) else 0.0
        # Nociception excess: BIS < 60 with elevated EMG (muscle artifact = pain response)
        if bis_val < config["bis_nociception_threshold"] and emg_val > 30:
            labels[t] = 1
        # Hypnosis excess: BIS > 65 AND rising trend (agent wash-in)
        elif bis_val > config["bis_hypnosis_threshold"]:
            past_bis = bis[t - window: t]
            past_valid = past_bis[~np.isnan(past_bis)]
            if len(past_valid) > 0 and bis_val > np.nanmean(past_valid) * 1.10:
                labels[t] = 2
    return labels

# PREPROCESSING UTILITIES

In [16]:
def forward_fill_gaps(arr: np.ndarray, max_gap_sec: int = 60) -> np.ndarray:
    """Forward-fill NaN gaps up to max_gap_sec; backward-fill any leading NaNs.
    Parameters
    ---------
    arr : np.ndarray  shape (T, F) or (T,)
    max_gap_sec : int   maximum gap in seconds (== samples at 1 Hz)
    Returns
    ------
    np.ndarray  filled array of same shape
    """
    is_1d = arr.ndim == 1
    if is_1d:
        arr = arr[:, np.newaxis]
    out = arr.copy().astype(np.float32)
    T, F = out.shape
    for f in range(F):
        col = out[:, f]
        # Forward fill with max_gap cap
        last_valid = None
        last_valid_idx = -1
        for t in range(T):
            if not np.isnan(col[t]):
                last_valid = col[t]
                last_valid_idx = t
            elif last_valid is not None and (t - last_valid_idx) <= max_gap_sec:
                col[t] = last_valid
        # Backward fill leading NaNs
        first_valid = None
        for t in range(T):
            if not np.isnan(col[t]):
                first_valid = col[t]
                break
        if first_valid is not None:
            for t in range(T):
                if np.isnan(col[t]):
                    col[t] = first_valid
                else:
                    break
        out[:, f] = col
    return out[:, 0] if is_1d else out

In [17]:
def remove_outliers_eeg(arr: np.ndarray, clip_uv: float = 500.0) -> np.ndarray:
    """Clip raw EEG to ±clip_uv (physiological plausibility: >500 μV is artifact).
    Parameters
    ---------
    arr : np.ndarray   raw EEG (T, 2)
    clip_uv : float    clipping threshold in μV
    Returns
    ------
    np.ndarray
    """
    return np.clip(arr, -clip_uv, clip_uv)

In [18]:
def remove_outliers_bis(arr: np.ndarray) -> np.ndarray:
    """Clip BIS scalar channels to physiologically valid ranges.
    Parameters
    ---------
    arr : np.ndarray  (T, 6): BIS, SQI, EMG, SR, SEF, TOTPOW
    Returns
    ------
    np.ndarray
    """
    out = arr.copy().astype(np.float32)
    clip_ranges = [(0, 100), (0, 100), (0, 100), (0, 100), (0, 50), (0, None)]
    for i, (lo, hi) in enumerate(clip_ranges):
        if hi is not None:
            out[:, i] = np.clip(out[:, i], lo, hi)
        else:
            out[:, i] = np.clip(out[:, i], lo, np.inf)
    return out

# SPECTRAL FEATURE COMPUTATION (EEG-specific)

In [19]:
def compute_band_power(
    psd: np.ndarray, freqs: np.ndarray, fmin: float, fmax: float
) -> float:
    """Integrate PSD within a frequency band using trapezoidal rule.
    Parameters
    ---------
    psd : np.ndarray   power spectral density
    freqs : np.ndarray frequency axis
    fmin, fmax : float band boundaries
    Returns
    ------
    float  band power
    """
    mask = (freqs >= fmin) & (freqs <= fmax)
    return float(np.trapz(psd[mask], freqs[mask])) if mask.sum() > 1 else 0.0


In [20]:
def compute_sef(
    psd: np.ndarray, freqs: np.ndarray, percentile: float = 95.0
) -> float:
    """Spectral Edge Frequency — frequency below which X% of power resides.
    Parameters
    ---------
    psd : np.ndarray
    freqs : np.ndarray
    percentile : float   default 95
    Returns
    ------
    float  SEF in Hz
    """
    cumpower = np.cumsum(psd)
    if cumpower[-1] == 0:
        return 0.0
    threshold = (percentile / 100.0) * cumpower[-1]
    idx = np.searchsorted(cumpower, threshold)
    idx = min(idx, len(freqs) - 1)
    return float(freqs[idx])

In [21]:
def compute_bsr(eeg_chunk: np.ndarray, fs: float = 128.0,
                window_sec: float = 63.0, amp_threshold: float = 5.0) -> float:
    """Burst Suppression Ratio: fraction of window with amplitude < threshold.
    Clinically: BSR > 0 implies deep anesthesia / barbiturate coma.
    Parameters
    ---------
    eeg_chunk : np.ndarray  1D raw EEG
    fs : float              sampling rate
    window_sec : float      analysis window in seconds
    amp_threshold : float   μV threshold
    Returns
    ------
    float  BSR in [0, 1]
    """
    n = int(window_sec * fs)
    if len(eeg_chunk) < n:
        return 0.0
    chunk = eeg_chunk[-n:]
    suppressed = np.abs(chunk) < amp_threshold
    return float(suppressed.mean())

In [22]:
def compute_spectral_features(
    eeg_window: np.ndarray, fs: float = 128.0, config: Dict = CONFIG
) -> np.ndarray:
    """Compute spectral feature vector for one 30-s EEG window.
    Features per channel: delta, theta, alpha, beta power + SEF95 + BSR = 6 each.
    Total: 2 channels × 6 = 12 features.
    Parameters
    ---------
    eeg_window : np.ndarray  shape (15000, 2) — 30 s at 500 Hz
    fs : float               sampling rate
    config : Dict
    Returns
    ------
    np.ndarray  float32 shape (12,)
    """
    stft_win = int(config["stft_window_sec"] * fs)
    noverlap = int(stft_win * config["stft_overlap"])
    features = []
    for ch in range(eeg_window.shape[1]):
        x = eeg_window[:, ch]
        if np.all(np.isnan(x)):
            features.extend([0.0] * 6)
            continue
        x = np.nan_to_num(x, nan=0.0)
        # Compute PSD via Welch method (more stable than single STFT for short clips)
        freqs, psd = scipy_signal.welch(
            x, fs=fs, nperseg=stft_win, noverlap=noverlap, window="hann"
        )
        band_powers = [
            compute_band_power(psd, freqs, lo, hi)
            for lo, hi in config["bands"].values()
        ]
        sef = compute_sef(psd, freqs, config["sef_percentile"])
        bsr = compute_bsr(x, fs=fs,
                          window_sec=config["bsr_window_sec"],
                          amp_threshold=config["bsr_amplitude_threshold_uv"])
        features.extend(band_powers + [sef, bsr])
    return np.array(features, dtype=np.float32)  # (12,)

# WINDOW GENERATION

In [23]:
def generate_windows(
    case_data: Dict[int, Dict],
    ioh_labels: Dict[int, Dict[str, np.ndarray]],
    nh_labels: Dict[int, np.ndarray],
    config: Dict,
) -> List[Dict]:
    """Slide windows over all eligible cases and collect feature + label dicts.
    Parameters
    ---------
    case_data : dict  caseid → {'eeg': ..., 'bis': ..., 'map': ...}
    ioh_labels : dict caseid → {'ioh_5': ..., 'ioh_10': ..., 'ioh_15': ...}
    nh_labels  : dict caseid → np.ndarray
    config : Dict
    Returns
    ------
    list of dicts, each representing one training window
    """
    windows = []
    history = config["history_sec"]
    stride = config["stride_sec"]
    eeg_win = config["eeg_samples_per_window"]  # 3840 samples = 30 s
    nan_thr = config["nan_threshold"]
    fs = config["fs_eeg"]
    for caseid, data in tqdm(case_data.items(), desc="Generating windows"):
        bis = data.get("bis")
        eeg = data.get("eeg")
        if bis is None:
            continue

        T_1hz = bis.shape[0]
        ioh_case = ioh_labels.get(caseid, {})
        nh_case = nh_labels.get(caseid, np.zeros(T_1hz, dtype=np.int64))

        for t_end in range(history, T_1hz, stride):
            t_start = t_end - history

            # ── BIS scalars window ──────────────────────────────────────────
            bis_win = bis[t_start:t_end]  # (1800, 6)

            # Discard window if >20% NaN in any BIS channel
            nan_frac = np.isnan(bis_win).mean(axis=0).max()
            if nan_frac > nan_thr:
                continue

            # SQI column for this window
            sqi_col = bis_win[:, 1]
            sqi_valid = (sqi_col >= config["sqi_hard_gate"]).astype(np.float32)
            modality_present = bool(sqi_valid.mean() > 0.3)

            # ── EEG chunk: last 30 s of window at 128 Hz ───────────────────
            eeg_start_128 = (t_end - config["eeg_window_sec"]) * fs
            eeg_end_128 = t_end * fs
            spectral_feats = np.zeros(12, dtype=np.float32)

            if eeg is not None and eeg_end_128 <= eeg.shape[0]:
                eeg_chunk = eeg[int(eeg_start_128): int(eeg_end_128)]  # (3840, 2)
                if eeg_chunk.shape[0] == eeg_win:
                    nan_frac_eeg = np.isnan(eeg_chunk).mean()
                    if nan_frac_eeg <= nan_thr:
                        eeg_chunk = np.nan_to_num(eeg_chunk, nan=0.0)
                        spectral_feats = compute_spectral_features(eeg_chunk, fs, config)
            else:
                eeg_chunk = np.zeros((eeg_win, 2), dtype=np.float32)

            # ── Labels ──────────────────────────────────────────────────────
            label_t = t_end - 1  # label at last second of window

            windows.append({
                "caseid": caseid,
                "window_start": t_start,
                "window_end": t_end,
                "bis_window": bis_win.astype(np.float32),    # (1800, 6)
                "eeg_chunk": eeg_chunk.astype(np.float32),   # (3840, 2)
                "spectral_feats": spectral_feats,             # (12,)
                "sqi_flag": sqi_valid,                        # (1800,)
                "label_ioh_5":  float(ioh_case.get("ioh_5",  np.zeros(T_1hz))[label_t]),
                "label_ioh_10": float(ioh_case.get("ioh_10", np.zeros(T_1hz))[label_t]),
                "label_ioh_15": float(ioh_case.get("ioh_15", np.zeros(T_1hz))[label_t]),
                "label_nh":     int(nh_case[label_t]),
                "modality_present": modality_present,
            })

    logger.info(f"Generated {len(windows)} windows total.")
    return windows

# SCALER FITTING / APPLICATION

In [24]:
def fit_scalers(train_windows: List[Dict]) -> Dict[str, StandardScaler]:
    """Fit StandardScaler on training windows only — no data leakage.
    Parameters
    ---------
    train_windows : list of window dicts
    Returns
    ------
    dict of fitted scalers for 'bis' and 'spectral' features
    """
    # BIS scalars: (N × 1800, 6)
    bis_all = np.concatenate([w["bis_window"].reshape(-1, 6) for w in train_windows], axis=0)
    bis_scaler = StandardScaler()
    bis_scaler.fit(bis_all)

    # Spectral features: (N, 12)
    spec_all = np.stack([w["spectral_feats"] for w in train_windows], axis=0)
    spec_scaler = StandardScaler()
    spec_scaler.fit(spec_all)
    logger.info("Scalers fitted on training set.")
    return {"bis": bis_scaler, "spectral": spec_scaler}

In [25]:
def apply_scalers(windows: List[Dict], scalers: Dict[str, StandardScaler]) -> List[Dict]:
    """Apply fitted scalers to a window list in-place.
    Parameters
    ---------
    windows : list of window dicts
    scalers : dict from fit_scalers()
    Returns
    ------
    list of window dicts with scaled values
    """
    bis_scaler = scalers["bis"]
    spec_scaler = scalers["spectral"]

    for w in windows:
        T = w["bis_window"].shape[0]
        w["bis_window"] = bis_scaler.transform(
            w["bis_window"].reshape(-1, 6)
        ).reshape(T, 6).astype(np.float32)
        w["spectral_feats"] = spec_scaler.transform(
            w["spectral_feats"].reshape(1, -1)
        ).reshape(-1).astype(np.float32)

    return windows

# TRAIN / VAL / TEST SPLIT

In [26]:
def split_cases(
    all_caseids: List[int],
    val_frac: float = 0.10,
    test_frac: float = 0.10,
    seed: int = 42,
) -> Tuple[List[int], List[int], List[int]]:
    """Patient-level split — no case appears across multiple splits.
    Parameters
    ---------
    all_caseids : list of int
    val_frac : float
    test_frac : float
    seed : int
    Returns
    ------
    (train_ids, val_ids, test_ids)
    """
    train_val, test = train_test_split(all_caseids, test_size=test_frac, random_state=seed)
    val_size = val_frac / (1 - test_frac)
    train, val = train_test_split(train_val, test_size=val_size, random_state=seed)
    logger.info(f"Split: train={len(train)}, val={len(val)}, test={len(test)}")

    return train, val, test


# DATASET

In [27]:
class EEGDataset(Dataset):
    """PyTorch Dataset for EEG windows.
    Each item yields BIS scalars (1800, 6), raw EEG (3840, 2), spectral features (12,),
    and multi-task labels for IOH / N-H / SQI gating.
    Parameters
    ---------
    windows : list of window dicts
    mode : str   'pretrain' | 'supervised'
    config : Dict
    """
    def __init__(self, windows: List[Dict], mode: str = "supervised", config: Dict = CONFIG):
        self.windows = windows
        self.mode = mode
        self.config = config

    def __len__(self) -> int:
        return len(self.windows)

    def __getitem__(self, idx: int) -> Dict:
        w = self.windows[idx]
        bis = torch.from_numpy(w["bis_window"])          # (1800, 6)
        eeg = torch.from_numpy(w["eeg_chunk"])            # (3840, 2)
        spec = torch.from_numpy(w["spectral_feats"])      # (12,)
        sqi  = torch.from_numpy(w["sqi_flag"])            # (1800,)
        item = {
            "bis":            bis,
            "eeg":            eeg,
            "spectral_feats": spec,
            "sqi_flag":       sqi,
            "caseid":         w["caseid"],
            "window_start":   w["window_start"],
            "modality_present": torch.tensor(w["modality_present"], dtype=torch.bool),
            # Multi-task labels (always included, used selectively per mode)
            "label_ioh_5":  torch.tensor(w["label_ioh_5"],  dtype=torch.float32),
            "label_ioh_10": torch.tensor(w["label_ioh_10"], dtype=torch.float32),
            "label_ioh_15": torch.tensor(w["label_ioh_15"], dtype=torch.float32),
            "label_nh":     torch.tensor(w["label_nh"],     dtype=torch.long),
            "label_et":     torch.tensor(-1.0,              dtype=torch.float32),  # unavailable here
        }

        # ── MSM pretraining: add mask for model to predict ──────────────────
        if self.mode == "pretrain":
            item["eeg_masked"], item["mask"] = self._apply_msm_mask(eeg)
        return item

    def _apply_msm_mask(
        self, eeg: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Apply Masked Segment Modeling (MSM) mask to raw EEG.
        Randomly masks 20% of the signal in 0.5-s (250 sample) spans.
        Parameters
        ---------
        eeg : torch.Tensor  (3840, 2)
        Returns
        ------
        (masked_eeg, mask) both shape (3840, 2)
        """
        T, C = eeg.shape
        span = self.config["msm_span_samples"]
        target_mask_n = int(T * self.config["msm_mask_frac"])
        mask = torch.zeros(T, C, dtype=torch.bool)
        masked = eeg.clone()
        masked_count = 0
        while masked_count < target_mask_n:
            start = random.randint(0, T - span)
            mask[start: start + span] = True
            masked[start: start + span] = 0.0  # zero-mask (not learnable token for speed)
            masked_count += span

        return masked, mask

# DATALOADERS


In [28]:
def build_dataloaders(
    train_ds: Dataset,
    val_ds: Dataset,
    test_ds: Dataset,
    config: Dict,
    mode: str = "supervised",
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    """Build DataLoaders for all three splits.
    Parameters
    ---------
    train_ds, val_ds, test_ds : Dataset
    config : Dict
    mode : str  'pretrain' | 'supervised'
    Returns
    ------
    (train_loader, val_loader, test_loader)
    """
    bs = config["msm_batch_size"] if mode == "pretrain" else config["finetune_batch_size"]
    nw = config["num_workers"]
    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True,
                              num_workers=nw, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=bs, shuffle=False,
                              num_workers=nw, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=bs, shuffle=False,
                              num_workers=nw, pin_memory=True)
    return train_loader, val_loader, test_loader

# MODEL ARCHITECTURE

## WaveNet Residual Block

In [29]:
class WaveNetBlock(nn.Module):
    """Single dilated causal residual block from WaveNet (van den Oord et al., 2016).
    Uses gated activation (tanh ⊙ sigmoid) + residual + skip connections.
    Causal padding ensures no future information leaks into predictions.
    Parameters
    ---------
    residual_channels : int
    skip_channels : int
    kernel_size : int
    dilation : int
    """
    def __init__(
        self,
        residual_channels: int,
        skip_channels: int,
        kernel_size: int,
        dilation: int,
    ):
        super().__init__()
        # Causal padding: (kernel_size-1)*dilation zeros prepended on left only
        self.causal_pad = (kernel_size - 1) * dilation
        self.dilated_conv = nn.Conv1d(
            in_channels=residual_channels,
            out_channels=residual_channels * 2,  # split into tanh + sigmoid gates
            kernel_size=kernel_size,
            dilation=dilation,
            padding=0,  # we handle padding manually for strict causality
        )
        self.residual_conv = nn.Conv1d(residual_channels, residual_channels, kernel_size=1)
        self.skip_conv     = nn.Conv1d(residual_channels, skip_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Parameters
        ---------
        x : (B, C, T)
        Returns
        ------
        residual : (B, C, T)
        skip     : (B, skip_channels, T)
        """
        # Left-only causal padding prevents future leakage
        h = F.pad(x, (self.causal_pad, 0))
        h = self.dilated_conv(h)
        # Gated activation: tanh(z1) ⊙ sigmoid(z2)
        z_tanh, z_sig = h.chunk(2, dim=1)
        h = torch.tanh(z_tanh) * torch.sigmoid(z_sig)
        skip     = self.skip_conv(h)
        residual = self.residual_conv(h) + x  # residual connection
        return residual, skip


## WaveNet Encoder (16 layers, 2 dilation cycles)

In [30]:
class WaveNetEncoder(nn.Module):
    """16-layer WaveNet dilated causal CNN for raw EEG feature extraction.
    Input:  (B, 2, 3840)   — 30 s of dual-channel EEG at 500 Hz
    Output: (B, skip_channels, T_out)   — compressed feature representation
    Parameters
    ---------
    in_channels : int        raw EEG channels (2)
    residual_channels : int  64
    skip_channels : int      64
    kernel_size : int        3
    dilations : list[int]    [1,2,...,128, 1,2,...,128]
    """
    def __init__(
        self,
        in_channels: int = 2,
        residual_channels: int = 64,
        skip_channels: int = 64,
        kernel_size: int = 3,
        dilations: List[int] = None,
    ):
        super().__init__()
        if dilations is None:
            dilations = CONFIG["wavenet_dilations"]
        # Causal input projection
        self.input_conv = nn.Conv1d(in_channels, residual_channels, kernel_size=1)
        self.blocks = nn.ModuleList([
            WaveNetBlock(residual_channels, skip_channels, kernel_size, d)
            for d in dilations
        ])
        # Output stack aggregates all skip connections
        self.output_proj = nn.Sequential(
            nn.ReLU(),
            nn.Conv1d(skip_channels, skip_channels, kernel_size=1),
            nn.ReLU(),
            nn.Conv1d(skip_channels, skip_channels, kernel_size=1),
        )


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ---------
        x : (B, 2, 15000)
        Returns
        ------
        out : (B, skip_channels, T_out)
        """
        h = self.input_conv(x)
        skip_sum = torch.zeros_like(h[:, :self.blocks[0].skip_conv.out_channels, :])
        for block in self.blocks:
            h, skip = block(h)
            # All skip outputs are the same length T (causal padding preserves length)
            skip_sum = skip_sum + skip
        return self.output_proj(skip_sum)   # (B, 64, T_out)

## Sinusoidal Positional Encoding

In [31]:
class SinusoidalPositionalEncoding(nn.Module):
    """Classic fixed sinusoidal positional encoding (Vaswani et al., 2017).
    Parameters
    ---------
    d_model : int
    max_len : int
    dropout : float
    """
    def __init__(self, d_model: int, max_len: int = 2048, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, d_model)"""
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

## Main EEG Model

In [32]:
class EEGModel(nn.Module):
    """Full EEG specialist encoder: WaveNet → Transformer → 128-dim embedding.
    Pipeline per forward pass:
        1. WaveNet CNN processes raw EEG → compressed feature sequence
        2. Temporal mean pooling → 512 × 64-dim (downsampled to match 1-Hz cadence)
        3. Spectral features + BIS scalars projected and concatenated
        4. 4-layer Transformer encoder with sinusoidal PE
        5. Final projection → 128-dim embedding per timestep
    MSM pretraining head:
        - Decoder reconstructs masked EEG segments from CNN features
        - Disabled during supervised fine-tuning
    Parameters
    ---------
    config : Dict
    """
    def __init__(self, config: Dict = CONFIG):
        super().__init__()
        self.config = config
        d_model = config["d_model"]          # 128
        fs = config["fs_eeg"]                # 128
        eeg_samples = config["eeg_samples_per_window"]  # 3840
        # ── WaveNet backbone ────────────────────────────────────────────────
        self.wavenet = WaveNetEncoder(
            in_channels=2,
            residual_channels=config["wavenet_residual_channels"],
            skip_channels=config["wavenet_skip_channels"],
            kernel_size=config["wavenet_kernel_size"],
            dilations=config["wavenet_dilations"],
        )
        # Adaptive pool to compress (B, 64, 15000) → (B, 64, 512)
        self.cnn_pool = nn.AdaptiveAvgPool1d(config["wavenet_output_length"])
        # ── Feature projections ─────────────────────────────────────────────
        # CNN features: 64 → d_model component
        self.cnn_proj = nn.Linear(64, d_model // 2)  # 64 → 64
        # Spectral features: 12 → d_model component
        self.spec_proj = nn.Linear(12, d_model // 4)  # 12 → 32
        # BIS scalar summary (global mean over time window): 6 → d_model component
        self.bis_proj = nn.Linear(6, d_model // 4)   # 6 → 32
        # Combined projection to d_model
        # d_model//2 + d_model//4 + d_model//4 = d_model
        self.input_proj = nn.Linear(d_model, d_model)
        # ── Transformer encoder ──────────────────────────────────────────────
        self.pos_enc = SinusoidalPositionalEncoding(d_model, max_len=600,
                                                    dropout=config["dropout"])
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=config["n_heads"],
            dim_feedforward=config["feedforward_dim"],
            dropout=config["dropout"],
            batch_first=True,
            norm_first=True,  # Pre-LN for training stability
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=config["transformer_layers"])
        # ── Output heads ────────────────────────────────────────────────────
        # Embedding projection: d_model → 128
        self.embedding_head = nn.Linear(d_model, 128)
        # IOH classification heads (one per horizon)
        self.ioh_head_5  = nn.Linear(128, 1)
        self.ioh_head_10 = nn.Linear(128, 1)
        self.ioh_head_15 = nn.Linear(128, 1)
        # N/H 3-class head
        self.nh_head = nn.Linear(128, 3)
        # ── MSM reconstruction head ─────────────────────────────────────────
        # Reconstructs masked EEG from CNN features: 64 → 2 channels × span
        self.msm_decoder = nn.Sequential(
            nn.Linear(64, 128),
            nn.GELU(),
            nn.Linear(128, 2),  # reconstruct both EEG channels point-wise
        )

    def encode_eeg(
        self,
        eeg: torch.Tensor,
        spec: torch.Tensor,
        bis: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Encode EEG + BIS scalars into 128-dim embedding sequence.
        Parameters
        ---------
        eeg  : (B, 3840, 2)   raw EEG window
        spec : (B, 12)          spectral features
        bis  : (B, 1800, 6)    BIS scalars for full history
        Returns
        ------
        embedding : (B, seq_len, 128)
        cnn_feats : (B, 64, 512)  — needed for MSM decoder
        """
        B = eeg.size(0)
        # WaveNet: (B, 2, 3840) → (B, 64, T_cnn)
        eeg_t = eeg.permute(0, 2, 1)          # channel-first for Conv1d
        cnn_out = self.wavenet(eeg_t)          # (B, 64, 3840)
        cnn_out = self.cnn_pool(cnn_out)       # (B, 64, 512) — downsample to 512 steps
        # Project CNN features: (B, 512, 64) → (B, 512, d_model/2)
        cnn_feats_t = cnn_out.permute(0, 2, 1)          # (B, 512, 64)
        cnn_proj = self.cnn_proj(cnn_feats_t)            # (B, 512, 64)
        # Spectral features broadcast across sequence: (B, 12) → (B, 512, 32)
        spec_proj = self.spec_proj(spec).unsqueeze(1).expand(-1, 512, -1)
        # BIS summary: global mean over 1800-step history → (B, 6) → (B, 512, 32)
        bis_summary = bis.mean(dim=1)                    # (B, 6)
        bis_proj = self.bis_proj(bis_summary).unsqueeze(1).expand(-1, 512, -1)
        # Concatenate and project to d_model
        combined = torch.cat([cnn_proj, spec_proj, bis_proj], dim=-1)  # (B, 512, 128)
        x = self.input_proj(combined)                                   # (B, 512, 128)
        # Positional encoding + Transformer
        x = self.pos_enc(x)
        x = self.transformer(x)                          # (B, 512, d_model)
        # Final embedding head
        embedding = self.embedding_head(x)               # (B, 512, 128)
        return embedding, cnn_out  # return cnn_out for MSM decoder access

    def forward(
        self,
        eeg: torch.Tensor,
        spec: torch.Tensor,
        bis: torch.Tensor,
        sqi_flag: Optional[torch.Tensor] = None,
    ) -> Dict[str, torch.Tensor]:
        """Supervised forward pass.
        Parameters
        ---------
        eeg      : (B, 3840, 2)
        spec     : (B, 12)
        bis      : (B, 1800, 6)
        sqi_flag : (B, T) — 1.0 where SQI ≥ 70
        Returns
        ------
        dict with keys: 'embedding', 'ioh_5', 'ioh_10', 'ioh_15', 'nh'
        """
        embedding, _ = self.encode_eeg(eeg, spec, bis)
        # Use the last timestep's embedding for scalar predictions
        # Clinically: the most recent state is the label-aligned timestep
        last_emb = embedding[:, -1, :]     # (B, 128)
        # Apply SQI zero-weighting: if SQI < 50, zero out EEG contribution
        if sqi_flag is not None:
            last_sqi = sqi_flag[:, -1]     # (B,) — last timestep SQI
            zero_mask = (last_sqi < (CONFIG["sqi_zero_weight"] / 100.0)).float()
            last_emb = last_emb * (1 - zero_mask.unsqueeze(-1))
        ioh_5  = torch.sigmoid(self.ioh_head_5(last_emb)).squeeze(-1)   # (B,)
        ioh_10 = torch.sigmoid(self.ioh_head_10(last_emb)).squeeze(-1)
        ioh_15 = torch.sigmoid(self.ioh_head_15(last_emb)).squeeze(-1)
        nh     = self.nh_head(last_emb)                                  # (B, 3)
        return {
            "embedding": last_emb,   # 128-dim — used by fusion model
            "ioh_5":     ioh_5,
            "ioh_10":    ioh_10,
            "ioh_15":    ioh_15,
            "nh":        nh,
        }

    def forward_msm(
        self,
        eeg_masked: torch.Tensor,
        spec: torch.Tensor,
        bis: torch.Tensor,
        mask: torch.Tensor
    ) -> torch.Tensor:
        """MSM pretraining forward pass — reconstruct masked EEG segments.
        Parameters
        ---------
        eeg_masked : (B, 3840, 2)   EEG with masked spans zeroed out
        spec       : (B, 12)
        bis        : (B, 1800, 6)
        mask       : (B, 3840, 2)   bool, True where masked
        Returns
        ------
        recon_loss : scalar tensor
        """
        # Run encoder on masked input
        _, cnn_feats = self.encode_eeg(eeg_masked, spec, bis)  # cnn_feats: (B, 64, 512)
        # Upsample CNN features back to original EEG chunk resolution
        cnn_upsampled = F.interpolate(cnn_feats, size=self.config["eeg_samples_per_window"], mode="linear",
                                      align_corners=False)   # (B, 64, 3840)
        cnn_upsampled = cnn_upsampled.permute(0, 2, 1)       # (B, 3840, 64)
        # Reconstruct 2-channel EEG at every masked position
        recon = self.msm_decoder(cnn_upsampled)               # (B, 3840, 2)
        return recon, mask

# LOSS FUNCTIONS

In [33]:
def msm_reconstruction_loss(
    recon: torch.Tensor,
    target: torch.Tensor,
    mask: torch.Tensor,
) -> torch.Tensor:
    """MSE loss only over masked positions.
    Parameters
    ---------
    recon  : (B, T, 2)   model reconstruction
    target : (B, T, 2)   original clean EEG
    mask   : (B, T, 2)   bool, True where masked
    Returns
    ------
    scalar tensor
    """
    loss = F.mse_loss(recon[mask], target[mask])
    return loss

In [34]:
def ioh_bce_loss(
    pred: torch.Tensor,
    label: torch.Tensor,
    sqi_weight: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    """Binary cross-entropy for IOH prediction with optional SQI-based weighting.
    Parameters
    ---------
    pred        : (B,) probabilities
    label       : (B,) binary 0/1
    sqi_weight  : (B,) float weight per sample
    Returns
    ------
    scalar tensor
    """
    loss = F.binary_cross_entropy(pred, label, weight=sqi_weight, reduction="mean")
    return loss

In [35]:
class FocalLoss(nn.Module):
    """Focal loss for imbalanced N/H classification (Lin et al., 2017).
    Down-weights easy negatives via (1-p_t)^gamma factor,
    allowing model to focus on hard nociception/hypnosis boundaries.
    Parameters
    ---------
    gamma : float   focusing parameter (2.0 recommended)
    alpha : Optional[torch.Tensor]  class-frequency inverse weights
    label_smoothing : float
    """
    def __init__(
        self,
        gamma: float = 2.0,
    ):
        alpha: Optional[torch.Tensor] = None,
        label_smoothing: float = 0.15,
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.label_smoothing = label_smoothing
    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ---------
        logits  : (B, 3)
        targets : (B,) long in {0, 1, 2}
        Returns
        ------
        scalar tensor
        """
        B, C = logits.shape
        # Label smoothing: spread probability mass to non-target classes
        with torch.no_grad():
            smooth_targets = torch.full((B, C), self.label_smoothing / (C - 1),
                                        device=logits.device)
            smooth_targets.scatter_(1, targets.unsqueeze(1),
                                    1.0 - self.label_smoothing)
        log_probs = F.log_softmax(logits, dim=-1)
        probs = torch.exp(log_probs)
        # Focal weight: (1 - p_target)^gamma
        p_target = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        focal_weight = (1 - p_target) ** self.gamma
        # Weighted cross-entropy with smooth targets
        ce = -(smooth_targets * log_probs).sum(dim=-1)
        loss = focal_weight * ce
        if self.alpha is not None:
            alpha_weight = self.alpha[targets]
            loss = alpha_weight * loss
        return loss.mean()

In [36]:
def compute_class_weights(labels: np.ndarray, n_classes: int = 3) -> torch.Tensor:
    """Compute inverse-frequency class weights for imbalanced N/H labels.
    Parameters
    ---------
    labels    : 1D array of class indices
    n_classes : int
    Returns
    ------
    torch.Tensor  shape (n_classes,)
    """
    counts = np.bincount(labels, minlength=n_classes).astype(float)
    counts = np.maximum(counts, 1.0)  # avoid zero division
    weights = 1.0 / counts
    weights = weights / weights.sum() * n_classes  # normalise
    return torch.tensor(weights, dtype=torch.float32)


# METRICS

In [37]:
def compute_metrics(
    preds: Dict[str, np.ndarray], labels: Dict[str, np.ndarray]
) -> Dict[str, float]:
    """Compute classification metrics for IOH and N/H tasks.
    Parameters
    ---------
    preds  : dict with keys 'ioh_5', 'ioh_10', 'ioh_15', 'nh'
    labels : dict with same keys
    Returns
    ------
    dict of metric name → float value
    """
    from sklearn.metrics import (
        roc_auc_score, average_precision_score,
        f1_score, accuracy_score
    )
    metrics: Dict[str, float] = {}
    for horizon in ["ioh_5", "ioh_10", "ioh_15"]:
        p = preds.get(horizon)
        l = labels.get(horizon)
        if p is None or l is None or len(np.unique(l)) < 2:
            continue
        metrics[f"auroc_{horizon}"] = roc_auc_score(l, p)
        metrics[f"auprc_{horizon}"] = average_precision_score(l, p)
        metrics[f"f1_{horizon}"]    = f1_score(l, (p > 0.5).astype(int),
                                               zero_division=0)
    if "nh" in preds and "nh" in labels:
        nh_pred = preds["nh"]  # class indices
        nh_true = labels["nh"]
        metrics["nh_accuracy"] = accuracy_score(nh_true, nh_pred)
        metrics["nh_f1_macro"] = f1_score(nh_true, nh_pred, average="macro",
                                          zero_division=0)
    return metrics

# TRAIN ONE EPOCH

In [38]:
def train_one_epoch_pretrain(
    model: EEGModel,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    amp_scaler: GradScaler,
    config: Dict,
) -> float:
    """One MSM pretraining epoch.
    Parameters
    ---------
    model, loader, optimizer, amp_scaler, config
    Returns
    ------
    float  mean MSM reconstruction loss
    """
    model.train()
    total_loss = 0.0
    n_batches = 0
    for batch in tqdm(loader, desc="Pretrain", leave=False):
        eeg_masked = batch["eeg_masked"].to(DEVICE)   # (B, 15000, 2)
        eeg_clean  = batch["eeg"].to(DEVICE)
        spec       = batch["spectral_feats"].to(DEVICE)
        bis        = batch["bis"].to(DEVICE)
        mask       = batch["mask"].to(DEVICE)
        optimizer.zero_grad()
        with autocast():
            recon, _ = model.forward_msm(eeg_masked, spec, bis, mask)
            loss = msm_reconstruction_loss(recon, eeg_clean, mask)
        amp_scaler.scale(loss).backward()
        amp_scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
        amp_scaler.step(optimizer)
        amp_scaler.update()
        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(n_batches, 1)

In [39]:
def train_one_epoch_supervised(
    model: EEGModel,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    amp_scaler: GradScaler,
    focal_loss_fn: FocalLoss,
    config: Dict,
) -> float:
    """One supervised fine-tuning epoch (IOH + N/H multi-task).
    Parameters
    ---------
    model, loader, optimizer, amp_scaler, focal_loss_fn, config
    Returns
    ------
    float  mean combined loss
    """
    model.train()
    total_loss = 0.0
    n_batches = 0
    for batch in tqdm(loader, desc="Train", leave=False):
        eeg  = batch["eeg"].to(DEVICE)
        spec = batch["spectral_feats"].to(DEVICE)
        bis  = batch["bis"].to(DEVICE)
        sqi  = batch["sqi_flag"].to(DEVICE)
        # SQI gate: weight each sample by proportion of valid SQI in window
        # Samples with low SQI are down-weighted (not excluded entirely, as they
        # still carry partial structural EEG information)
        sqi_sample_weight = sqi.mean(dim=1).clamp(min=0.1)  # (B,) — floor at 0.1
        l_5  = batch["label_ioh_5"].to(DEVICE)
        l_10 = batch["label_ioh_10"].to(DEVICE)
        l_15 = batch["label_ioh_15"].to(DEVICE)
        l_nh = batch["label_nh"].to(DEVICE)
        optimizer.zero_grad()
        with autocast():
            out = model(eeg, spec, bis, sqi)
            # IOH losses with SQI-gated weighting
            loss_ioh_5  = ioh_bce_loss(out["ioh_5"],  l_5,  sqi_sample_weight)
            loss_ioh_10 = ioh_bce_loss(out["ioh_10"], l_10, sqi_sample_weight)
            loss_ioh_15 = ioh_bce_loss(out["ioh_15"], l_15, sqi_sample_weight)
            loss_ioh = loss_ioh_5 + loss_ioh_10 + loss_ioh_15
            # N/H focal loss
            loss_nh = focal_loss_fn(out["nh"], l_nh)
            # Combined: IOH is primary task for Model 3; N/H secondary
            loss = loss_ioh + 0.5 * loss_nh
        amp_scaler.scale(loss).backward()
        amp_scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
        amp_scaler.step(optimizer)
        amp_scaler.update()
        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(n_batches, 1)

# VALIDATION

In [40]:
def validate(
    model: EEGModel,
    loader: DataLoader,
    focal_loss_fn: FocalLoss,
    config: Dict,
) -> Tuple[float, Dict[str, float]]:
    """Run validation pass and compute metrics.
    Parameters
    ---------
    model, loader, focal_loss_fn, config
    Returns
    ------
    (val_loss, metrics_dict)
    """
    model.eval()
    total_loss = 0.0
    n_batches = 0
    all_preds: Dict[str, List] = {k: [] for k in ["ioh_5", "ioh_10", "ioh_15", "nh"]}
    all_labels: Dict[str, List] = {k: [] for k in ["ioh_5", "ioh_10", "ioh_15", "nh"]}
    with torch.no_grad():
        for batch in tqdm(loader, desc="Val", leave=False):
            eeg  = batch["eeg"].to(DEVICE)
            spec = batch["spectral_feats"].to(DEVICE)
            bis  = batch["bis"].to(DEVICE)
            sqi  = batch["sqi_flag"].to(DEVICE)
            sqi_weight = sqi.mean(dim=1).clamp(min=0.1)
            l_5  = batch["label_ioh_5"].to(DEVICE)
            l_10 = batch["label_ioh_10"].to(DEVICE)
            l_15 = batch["label_ioh_15"].to(DEVICE)
            l_nh = batch["label_nh"].to(DEVICE)
            with autocast():
                out = model(eeg, spec, bis, sqi)
                loss_ioh = (
                    ioh_bce_loss(out["ioh_5"],  l_5,  sqi_weight) +
                    ioh_bce_loss(out["ioh_10"], l_10, sqi_weight) +
                    ioh_bce_loss(out["ioh_15"], l_15, sqi_weight)
                )
                loss_nh = focal_loss_fn(out["nh"], l_nh)
                loss = loss_ioh + 0.5 * loss_nh
            total_loss += loss.item()
            n_batches += 1
            all_preds["ioh_5"].extend(out["ioh_5"].cpu().numpy())
            all_preds["ioh_10"].extend(out["ioh_10"].cpu().numpy())
            all_preds["ioh_15"].extend(out["ioh_15"].cpu().numpy())
            all_preds["nh"].extend(out["nh"].argmax(dim=-1).cpu().numpy())
            all_labels["ioh_5"].extend(l_5.cpu().numpy())
            all_labels["ioh_10"].extend(l_10.cpu().numpy())
            all_labels["ioh_15"].extend(l_15.cpu().numpy())
            all_labels["nh"].extend(l_nh.cpu().numpy())
    for k in all_preds:
        all_preds[k] = np.array(all_preds[k])
        all_labels[k] = np.array(all_labels[k])
    metrics = compute_metrics(all_preds, all_labels)
    val_loss = total_loss / max(n_batches, 1)
    return val_loss, metrics

# EARLY STOPPING

In [41]:
class EarlyStopping:
    """Tracks best validation loss and triggers stop after patience epochs with no improvement.
    Parameters
    ---------
    patience : int   epochs without improvement before stopping
    min_delta : float  minimum improvement to count as better
    """
    def __init__(self, patience: int = 8, min_delta: float = 1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.counter = 0
        self.best_state: Optional[Dict] = None
    def step(self, val_loss: float, model: nn.Module) -> bool:
        """Update state. Returns True if training should stop.
        Parameters
        ---------
        val_loss : float
        model : nn.Module
        Returns
        ------
        bool  True → stop training
        """
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            logger.info(f"  → New best val_loss: {val_loss:.5f}")
        else:
            self.counter += 1
            logger.info(f"  EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                return True
        return False
    def restore_best(self, model: nn.Module) -> None:
        """Load best weights back into model.
        Parameters
        ---------
        model : nn.Module
        """
        if self.best_state is not None:
            model.load_state_dict(self.best_state)
            logger.info("Restored best model weights.")

# TRAIN MODEL (STAGE 0: MSM + STAGE 1: SUPERVISED)

In [42]:
def train_msm_stage(
    model: EEGModel,
    train_loader: DataLoader,
    val_loader: DataLoader,
    config: Dict,
) -> EEGModel:
    """Stage 0: Self-supervised MSM pretraining.
    No supervision signal needed — model learns EEG structure by
    reconstructing masked segments. Initialises strong CNN features
    before the supervised stage sees any labels.
    Parameters
    ---------
    model, train_loader, val_loader, config
    Returns
    ------
    EEGModel with pretrained CNN weights
    """
    logger.info("=" * 60)
    logger.info("STAGE 0 — MSM Self-Supervised Pretraining")
    logger.info("=" * 60)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["msm_lr"],
                                  weight_decay=config["weight_decay"])
    amp_scaler = GradScaler()
    early_stop = EarlyStopping(patience=5)
    for epoch in range(1, config["msm_epochs"] + 1):
        t0 = time.time()
        train_loss = train_one_epoch_pretrain(model, train_loader, optimizer,
                                              amp_scaler, config)
        # Validation: measure MSM loss on val set
        model.eval()
        val_loss = 0.0
        n = 0
        with torch.no_grad():
            for batch in val_loader:
                eeg_m = batch["eeg_masked"].to(DEVICE)
                eeg_c = batch["eeg"].to(DEVICE)
                sp    = batch["spectral_feats"].to(DEVICE)
                bi    = batch["bis"].to(DEVICE)
                mk    = batch["mask"].to(DEVICE)
                with autocast():
                    recon, _ = model.forward_msm(eeg_m, sp, bi, mk)
                    vl = msm_reconstruction_loss(recon, eeg_c, mk)
                val_loss += vl.item()
                n += 1
        val_loss /= max(n, 1)
        elapsed = time.time() - t0
        logger.info(f"MSM Epoch {epoch:3d}/{config['msm_epochs']} | "
                    f"train={train_loss:.5f}  val={val_loss:.5f}  "
                    f"({elapsed:.1f}s)")
        if early_stop.step(val_loss, model):
            logger.info("MSM early stopping triggered.")
            break
    early_stop.restore_best(model)
    torch.save(model.state_dict(), config["msm_checkpoint"])
    logger.info(f"MSM checkpoint saved → {config['msm_checkpoint']}")
    return model

In [43]:
def _plot_training_history(history: Dict, config: Dict) -> None:
    """Save training/validation loss curve to disk.
    Parameters
    ---------
    history : dict
    config : Dict
    """
    try:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        epochs = range(1, len(history["train_loss"]) + 1)
        axes[0].plot(epochs, history["train_loss"], label="train")
        axes[0].plot(epochs, history["val_loss"],   label="val")
        axes[0].set_title("Loss")
        axes[0].legend()
        axes[0].set_xlabel("Epoch")
        auroc_vals = [m.get("auroc_ioh_5", float("nan")) for m in history["metrics"]]
        nh_f1_vals = [m.get("nh_f1_macro",  float("nan")) for m in history["metrics"]]
        axes[1].plot(epochs, auroc_vals, label="AUROC IOH-5min")
        axes[1].plot(epochs, nh_f1_vals, label="N/H F1 macro")
        axes[1].set_title("Metrics")
        axes[1].legend()
        axes[1].set_xlabel("Epoch")
        plt.tight_layout()
        plt.savefig("model_3_training.png", dpi=150)
        plt.close()
        logger.info("Training plot saved → model_3_training.png")
    except Exception as e:
        logger.warning(f"Plot failed: {e}")

In [44]:
def train_model(
    model: EEGModel,
    train_loader: DataLoader,
    val_loader: DataLoader,
    train_windows: List[Dict],
    config: Dict,
) -> EEGModel:
    """Stage 1: Supervised fine-tuning for IOH + N/H prediction.
    Parameters
    ---------
    model, train_loader, val_loader, train_windows, config
    Returns
    ------
    EEGModel  best checkpoint restored
    """
    logger.info("=" * 60)
    logger.info("STAGE 1 — Supervised Fine-Tuning")
    logger.info("=" * 60)
    # Class weights for N/H focal loss (fitted on training labels only)
    nh_labels_train = np.array([w["label_nh"] for w in train_windows])
    class_weights = compute_class_weights(nh_labels_train, n_classes=3).to(DEVICE)
    focal_loss_fn = FocalLoss(
        gamma=config["nh_focal_gamma"],
        alpha=class_weights,
        label_smoothing=config["nh_label_smoothing"],
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["finetune_lr"],
                                  weight_decay=config["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=config["scheduler_patience"],
        factor=config["scheduler_factor"], verbose=True,
    )
    amp_scaler = GradScaler()
    early_stop = EarlyStopping(patience=config["early_stopping_patience"])
    history = {"train_loss": [], "val_loss": [], "metrics": []}
    for epoch in range(1, config["finetune_epochs"] + 1):
        t0 = time.time()
        train_loss = train_one_epoch_supervised(model, train_loader, optimizer,
                                                amp_scaler, focal_loss_fn, config)
        val_loss, metrics = validate(model, val_loader, focal_loss_fn, config)
        scheduler.step(val_loss)
        elapsed = time.time() - t0
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["metrics"].append(metrics)
        # Log key metrics
        auroc_5 = metrics.get("auroc_ioh_5", float("nan"))
        nh_f1   = metrics.get("nh_f1_macro",  float("nan"))
        logger.info(
            f"Epoch {epoch:3d}/{config['finetune_epochs']} | "
            f"train={train_loss:.4f}  val={val_loss:.4f}  "
            f"AUROC_IOH5={auroc_5:.3f}  NH_F1={nh_f1:.3f}  ({elapsed:.1f}s)"
        )
        if early_stop.step(val_loss, model):
            logger.info("Supervised early stopping triggered.")
            break
    early_stop.restore_best(model)
    torch.save(model.state_dict(), config["best_checkpoint"])
    logger.info(f"Best checkpoint saved → {config['best_checkpoint']}")
    _plot_training_history(history, config)
    return model

# Main Pipeline

In [45]:
"""End-to-end pipeline: load → preprocess → pretrain → fine-tune → evaluate."""
logger.info("Model 3 — EEG WaveNet+Transformer — starting pipeline")
logger.info(f"Device: {DEVICE}")



2026-06-11 17:41:28 [INFO] Model 3 — EEG WaveNet+Transformer — starting pipeline
INFO:model_3:Model 3 — EEG WaveNet+Transformer — starting pipeline
2026-06-11 17:41:28 [INFO] Device: cuda
INFO:model_3:Device: cuda


In [46]:
# ── Load clinical metadata ────────────────────────────────────────────────
clinical_df = load_clinical_metadata()

2026-06-11 17:41:28 [INFO] Clinical metadata loaded: 0 cases.
INFO:model_3:Clinical metadata loaded: 0 cases.


In [47]:
# ── Load waveform data ────────────────────────────────────────────────────
case_ids = list(range(CONFIG["case_id_range"][0], CONFIG["case_id_range"][1]))
if CONFIG["max_cases"] is not None:
    case_ids = case_ids[:CONFIG["max_cases"]]
    logger.info(f"Debug mode: capped at {CONFIG['max_cases']} cases.")

case_data: Dict[int, Dict] = {}
eligible_ids: List[int] = []

if VITALDB_AVAILABLE:
    for caseid in tqdm(case_ids, desc="Loading cases"):
        try:
            result = load_eeg_case(caseid, CONFIG)
        except Exception as e:
            logger.warning(f"caseid {caseid}: unexpected error — {e}")
            continue
        if result is None:
            logger.warning(f"caseid {caseid}: no usable data, skipping.")
            continue
        if not is_case_eligible(result, clinical_df, caseid, CONFIG):
            continue
        # Preprocess in-place
        if result["eeg"] is not None:
            result["eeg"] = remove_outliers_eeg(result["eeg"])
        if result["bis"] is not None:
            result["bis"] = remove_outliers_bis(result["bis"])
            result["bis"] = forward_fill_gaps(result["bis"], max_gap_sec=60)
        if result["map"] is not None:
            result["map"] = forward_fill_gaps(result["map"], max_gap_sec=60)

        case_data[caseid] = result
        eligible_ids.append(caseid)

    logger.info(f"Eligible cases loaded: {len(eligible_ids)}")

else:
    logger.warning("VitalDB unavailable — skipping data loading; using dummy data for demo.")
        # ── Minimal dummy data for dry-run / CI ──────────────────────────────
    for caseid in range(1, 21):
        T_1hz = 3600
        T_128hz = T_1hz * 128
        case_data[caseid] = {
            "eeg": np.random.randn(T_128hz, 2).astype(np.float32) * 20,
            "bis": np.random.rand(T_1hz, 6).astype(np.float32) * 100,
            "map": (np.random.rand(T_1hz) * 40 + 55).astype(np.float32),
            }
        eligible_ids.append(caseid)

    logger.info(f"Dummy dataset: {len(eligible_ids)} cases")



Loading cases: 100%|██████████| 199/199 [04:44<00:00,  1.43s/it]
2026-06-11 17:46:14 [INFO] Eligible cases loaded: 120
INFO:model_3:Eligible cases loaded: 120


In [48]:
# ── Build labels ──────────────────────────────────────────────────────────
logger.info("Building IOH and N/H labels...")
ioh_labels: Dict[int, Dict] = {}
nh_labels: Dict[int, np.ndarray] = {}
for caseid in tqdm(eligible_ids, desc="Labels"):
    map_series = case_data[caseid].get("map")
    bis_arr    = case_data[caseid].get("bis")
    if map_series is not None:
        ioh_labels[caseid] = build_ioh_labels(map_series, CONFIG)
    if bis_arr is not None:
        nh_labels[caseid] = build_nh_labels(bis_arr, CONFIG)



2026-06-11 17:46:14 [INFO] Building IOH and N/H labels...
INFO:model_3:Building IOH and N/H labels...
Labels: 100%|██████████| 120/120 [03:04<00:00,  1.53s/it]


In [49]:
# ── Generate windows ──────────────────────────────────────────────────────
logger.info("Generating training windows...")
all_windows = generate_windows(case_data, ioh_labels, nh_labels, CONFIG)
# Free raw case arrays from RAM — they are no longer needed
for caseid in list(case_data.keys()):
    del case_data[caseid]
gc.collect()
logger.info("Raw case data freed from RAM.")

if len(all_windows) == 0:
    logger.error("No windows generated — check data loading. Exiting.")


2026-06-11 17:49:18 [INFO] Generating training windows...
INFO:model_3:Generating training windows...
Generating windows:   0%|          | 0/120 [00:00<?, ?it/s]/tmp/ipykernel_21256/493767766.py:15: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(psd[mask], freqs[mask])) if mask.sum() > 1 else 0.0
Generating windows: 100%|██████████| 120/120 [01:27<00:00,  1.37it/s]
2026-06-11 17:50:46 [INFO] Generated 25133 windows total.
INFO:model_3:Generated 25133 windows total.
2026-06-11 17:50:46 [INFO] Raw case data freed from RAM.
INFO:model_3:Raw case data freed from RAM.


In [50]:
#  ── Split ─────────────────────────────────────────────────────────────────
train_ids, val_ids, test_ids = split_cases(eligible_ids, CONFIG["val_frac"], CONFIG["test_frac"])
train_id_set = set(train_ids)
val_id_set   = set(val_ids)
test_id_set  = set(test_ids)
train_wins = [w for w in all_windows if w["caseid"] in train_id_set]
val_wins   = [w for w in all_windows if w["caseid"] in val_id_set]
test_wins  = [w for w in all_windows if w["caseid"] in test_id_set]
logger.info(
    "Windows — train: %d, val: %d, test: %d",
    len(train_wins),
    len(val_wins),
    len(test_wins)
)




2026-06-11 17:50:46 [INFO] Split: train=96, val=12, test=12
INFO:model_3:Split: train=96, val=12, test=12
2026-06-11 17:50:46 [INFO] Windows — train: 19913, val: 2164, test: 3056
INFO:model_3:Windows — train: 19913, val: 2164, test: 3056


In [51]:
# ── Scalers ───────────────────────────────────────────────────────────────
scalers = fit_scalers(train_wins)
train_wins = apply_scalers(train_wins, scalers)
val_wins   = apply_scalers(val_wins,   scalers)
test_wins  = apply_scalers(test_wins,  scalers)

2026-06-11 17:50:55 [INFO] Scalers fitted on training set.
INFO:model_3:Scalers fitted on training set.


In [52]:
 # ── Stage 0: MSM Datasets + Loaders ──────────────────────────────────────
train_ds_msm = EEGDataset(train_wins, mode="pretrain", config=CONFIG)
val_ds_msm   = EEGDataset(val_wins,   mode="pretrain", config=CONFIG)
test_ds_msm  = EEGDataset(test_wins,  mode="pretrain", config=CONFIG)
train_loader_msm, val_loader_msm, _ = build_dataloaders(train_ds_msm, val_ds_msm, test_ds_msm, CONFIG, mode="pretrain")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [53]:
# ── Stage 1: Supervised Datasets + Loaders ────────────────────────────────
train_ds_sup = EEGDataset(train_wins, mode="supervised", config=CONFIG)
val_ds_sup   = EEGDataset(val_wins,   mode="supervised", config=CONFIG)
test_ds_sup  = EEGDataset(test_wins,  mode="supervised", config=CONFIG)
train_loader_sup, val_loader_sup, test_loader_sup = build_dataloaders(train_ds_sup, val_ds_sup, test_ds_sup, CONFIG, mode="supervised")

In [54]:
 # ── Build model ───────────────────────────────────────────────────────────
model = EEGModel(CONFIG).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
logger.info(f"Model parameters: {n_params:,}")

/tmp/ipykernel_21256/3835723060.py:53: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=config["transformer_layers"])
2026-06-11 17:51:06 [INFO] Model parameters: 1,113,992
INFO:model_3:Model parameters: 1,113,992


In [55]:
# ── Stage 0: MSM pretraining ──────────────────────────────────────────────
model = train_msm_stage(model, train_loader_msm, val_loader_msm, CONFIG)


2026-06-11 17:51:06 [INFO] ============================================================
INFO:model_3:============================================================
2026-06-11 17:51:06 [INFO] STAGE 0 — MSM Self-Supervised Pretraining
INFO:model_3:STAGE 0 — MSM Self-Supervised Pretraining
2026-06-11 17:51:06 [INFO] ============================================================
INFO:model_3:============================================================
/tmp/ipykernel_21256/2867400135.py:23: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  amp_scaler = GradScaler()
Pretrain:   0%|          | 0/1244 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker cr

KeyboardInterrupt: 

In [ ]:
 # ── Stage 1: Supervised fine-tuning ───────────────────────────────────────
model = train_model(model, train_loader_sup, val_loader_sup, train_wins, CONFIG)

In [ ]:
 # ── Final test set evaluation ─────────────────────────────────────────────
logger.info("=" * 60)
logger.info("FINAL EVALUATION on held-out test set")
logger.info("=" * 60)

nh_labels_train = np.array([w["label_nh"] for w in train_wins])
class_weights = compute_class_weights(nh_labels_train, 3).to(DEVICE)
focal_fn = FocalLoss(CONFIG["nh_focal_gamma"], class_weights, CONFIG["nh_label_smoothing"]).to(DEVICE)

test_loss, test_metrics = validate(model, test_loader_sup, focal_fn, CONFIG)
logger.info(f"Test loss: {test_loss:.4f}")
for k, v in sorted(test_metrics.items()):
    logger.info(f"  {k}: {v:.4f}")

In [ ]:
# ── Save final model ──────────────────────────────────────────────────────
torch.save({
        "model_state_dict": model.state_dict(),
        "config":           CONFIG,
        "test_metrics":     test_metrics,
    }, CONFIG["final_checkpoint"])

logger.info(f"Final model saved → {CONFIG['final_checkpoint']}")
logger.info("Model 3 pipeline complete.")